## Spark DataFrame Basics

Spark DataFrames allow for easy handling of large datasets.

* Easy syntax
* Ability to use SQL directly in the dataframe
* Operations can automatically distribute across RDDs

### Create a DataFrame

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('pyspark_basics').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 17:40:36 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/06/11 17:40:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 17:40:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
%%writefile user_simple.json
{"name": "Bob"}
{"name": "Jim", "age": 40}
{"name": "Mary", "age": 24}

Writing user_simple.json


In [9]:
df = spark.read.json("user_simple.json")

### Show DataFrame

In [10]:
df.show()

+----+----+
| age|name|
+----+----+
|NULL| Bob|
|  40| Jim|
|  24|Mary|
+----+----+



In [11]:
df.printSchema()

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)



In [12]:
df.columns

['age', 'name']

In [13]:
df.describe()

DataFrame[summary: string, age: string, name: string]

### Specifying Schema Structure

* Some data types make it easier to infer schema
* Often have to set the schema yourself
* Spark has tools to help specify the structure

Next, we need to create the list of Structure fields

* :param name: string, name of the field.
* :param dataType: class `DataType` of the field.
* :param nullable: boolean, whether the field can be null (None)

In [14]:
from pyspark.sql.types import StructField, StringType, IntegerType, StructType

In [15]:
data_schema = [StructField("age", IntegerType(), True), StructField("name", StringType(), True)]

In [16]:
final_struc = StructType(fields=data_schema)

In [17]:
df = spark.read.json("user_simple.json", schema=final_struc)

In [18]:
df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- name: string (nullable = true)



### Grab data

In [20]:
df['age']

Column<'age'>

In [21]:
type(df['age'])

pyspark.sql.classic.column.Column

In [22]:
df.select('age')

DataFrame[age: int]

In [23]:
type(df.select('age'))

pyspark.sql.classic.dataframe.DataFrame

In [24]:
df.select('age').show()

+----+
| age|
+----+
|NULL|
|  40|
|  24|
+----+



In [27]:
df.head(2)

[Row(age=None, name='Bob'), Row(age=40, name='Jim')]

In [28]:
df.select(['name', 'age'])

DataFrame[name: string, age: int]

In [29]:
df.select(['name', 'age']).show()

+----+----+
|name| age|
+----+----+
| Bob|NULL|
| Jim|  40|
|Mary|  24|
+----+----+



### Create New Columns

In [30]:
df.withColumn('newAge', df['age']).show()

+----+----+------+
| age|name|newAge|
+----+----+------+
|NULL| Bob|  NULL|
|  40| Jim|    40|
|  24|Mary|    24|
+----+----+------+



In [31]:
df.show()

+----+----+
| age|name|
+----+----+
|NULL| Bob|
|  40| Jim|
|  24|Mary|
+----+----+



In [32]:
df.withColumnRenamed('name', 'firstName').show()

+----+---------+
| age|firstName|
+----+---------+
|NULL|      Bob|
|  40|      Jim|
|  24|     Mary|
+----+---------+



In [33]:
df.show()

+----+----+
| age|name|
+----+----+
|NULL| Bob|
|  40| Jim|
|  24|Mary|
+----+----+



In [34]:
df.withColumn('agePlusTen', df['age']+10).show()

+----+----+----------+
| age|name|agePlusTen|
+----+----+----------+
|NULL| Bob|      NULL|
|  40| Jim|        50|
|  24|Mary|        34|
+----+----+----------+



In [35]:
df.withColumn('age_minus_5', df['age']-5).show()

+----+----+-----------+
| age|name|age_minus_5|
+----+----+-----------+
|NULL| Bob|       NULL|
|  40| Jim|         35|
|  24|Mary|         19|
+----+----+-----------+



### Using SQL

In [36]:
df.createOrReplaceTempView('customers')

In [37]:
sql_results = spark.sql("SELECT * from customers")

In [38]:
sql_results

DataFrame[age: int, name: string]

In [39]:
sql_results.show()

+----+----+
| age|name|
+----+----+
|NULL| Bob|
|  40| Jim|
|  24|Mary|
+----+----+



In [40]:
spark.sql('SELECT * FROM customers WHERE age=24').show()

+---+----+
|age|name|
+---+----+
| 24|Mary|
+---+----+

